In [ ]:
import requests
import pandas as pd
import json
import matplotlib.pyplot as plt

## EDA

In [ ]:
# Import des données source WorldPop_2020

df = pd.read_csv("ghana_villages_clean.csv")
df.head()

In [ ]:
# Description des données numériques
df.describe()

In [ ]:
# Nombre de valeurs manquantes par colonne
missing_counts = df.isna().sum()

print("Valeurs manquantes par colonne :")
print(missing_counts)



In [ ]:
df["Nearest_Facility_Type"].value_counts()

In [ ]:
nb_centres = df["Nearest_Facility_Name"].nunique()
nb_centres


In [ ]:
# Print all results for the value counts of 'Nearest_Facility_Name'
count=(df["Nearest_Facility_Name"].value_counts())
count[count > 1]


In [ ]:
df["Distance_Nearest_Facility_km"].describe()


In [ ]:
df_far = df[df["Distance_Nearest_Facility_km"] > 30]
df_far["Distance_Nearest_Facility_km"].describe()

In [ ]:
doublons = df["Village"].value_counts()
doublons[doublons > 1]


In [ ]:
nb_villages = df["Village"].nunique()
nb_villages


In [ ]:
df["Nearest_Facility_Name"].value_counts()


In [ ]:
df.groupby("Nearest_Facility_Name")["Village"].nunique().sort_values(ascending=False).head(10)


In [ ]:
df_centres = df[["Nearest_Facility_Name", "Nearest_Facility_Latitude", "Nearest_Facility_Longitude"]].drop_duplicates()

doublons = df_centres.groupby(["Nearest_Facility_Latitude", "Nearest_Facility_Longitude"]).size()
doublons[doublons > 1]


In [ ]:
# pip install geopandas

In [ ]:
import pandas as pd

zipline_data = {
    "Hub_ID": ["GH1", "GH2", "GH3", "GH4", "GH5", "GH6"],
    "Localisation": ["Omenako", "Mpanya", "Vobsi", "Sefwi Wiawso", "Anum", "Kete Krachi"],
    "Latitude": [6.2027, 7.0545, 10.3470, 6.1820, 6.4885, 7.7981],
    "Longitude": [-0.4462, -1.3855, -0.8035, -2.4845, 0.1625, -0.0125],
    "Region": ["Eastern", "Ashanti", "North East", "Western North", "Eastern / Volta", "Oti"]
}

df_zipline = pd.DataFrame(zipline_data)
df_zipline


In [ ]:
#pip install geopy

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from geopy.distance import geodesic
import pandas as pd

# ---------------------------------------------------------
# 1. Charger les régions du Ghana
# ---------------------------------------------------------
ghana_regions = gpd.read_file(
    "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_GHA_1.json"
)

# ---------------------------------------------------------
# 2. GeoDataFrame des villages éloignés (>30 km d'un centre)
# ---------------------------------------------------------
gdf_far = gpd.GeoDataFrame(
    df_far,
    geometry=gpd.points_from_xy(df_far.Longitude, df_far.Latitude),
    crs="EPSG:4326"
)

# ---------------------------------------------------------
# 3. GeoDataFrame des centres de santé les plus proches
# ---------------------------------------------------------
gdf_health = gpd.GeoDataFrame(
    df_far.copy(),  # on garde le lien village ↔ centre
    geometry=gpd.points_from_xy(
        df_far.Nearest_Facility_Longitude,
        df_far.Nearest_Facility_Latitude
    ),
    crs="EPSG:4326"
)

# ---------------------------------------------------------
# 4. GeoDataFrame des hubs Zipline
# ---------------------------------------------------------
gdf_zipline = gpd.GeoDataFrame(
    df_zipline,
    geometry=gpd.points_from_xy(df_zipline.Longitude, df_zipline.Latitude),
    crs="EPSG:4326"
)

# ---------------------------------------------------------
# 5. Calculer la distance centre → hub Zipline le plus proche
# ---------------------------------------------------------
def dist_health_to_nearest_hub(row, hubs_gdf):
    health_point = (row.Nearest_Facility_Latitude, row.Nearest_Facility_Longitude)

    dists = [
        geodesic(
            health_point,
            (hub.Latitude, hub.Longitude)
        ).km
        for _, hub in hubs_gdf.iterrows()
    ]
    return min(dists)

gdf_health["Dist_to_Nearest_Hub_km"] = gdf_health.apply(
    dist_health_to_nearest_hub,
    axis=1,
    hubs_gdf=gdf_zipline
)

# ---------------------------------------------------------
# 6. Filtrer les centres > 30 km d’un hub
#    Et filtrer les villages associés
# ---------------------------------------------------------
mask_far_hub = gdf_health["Dist_to_Nearest_Hub_km"] > 30

gdf_health_far_hub = gdf_health[mask_far_hub]
gdf_far_matched = gdf_far.loc[mask_far_hub]

# ---------------------------------------------------------
# 7. Tracer la carte finale
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 12))

# Contours du Ghana
ghana_regions.plot(ax=ax, color="white", edgecolor="black")

# Villages éloignés dont le centre est lui-même loin d'un hub
gdf_far_matched.plot(
    ax=ax,
    color="red",
    markersize=25,
    label="Villages >30 km d'un centre de santé"
) 
    
# Centres de santé >30 km d'un hub
gdf_health_far_hub.plot(
    ax=ax,
    color="green",
    markersize=40,
    marker="o",
    label="Centres de santé"
)

# Hubs Zipline
gdf_zipline.plot(
    ax=ax,
    color="purple",
    markersize=80,
    marker="^",
    label="Hubs Zipline"
)

plt.title("Villages éloignés, Centres de santé éloignés des hubs Zipline au Ghana")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# pip install mapclassify


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

# -----------------------------
# 1. Charger les régions du Ghana
# -----------------------------
admin1 = gpd.read_file(
    "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_1_states_provinces.zip"
)
ghana_regions = admin1[admin1["admin"] == "Ghana"].copy()

ghana_regions = ghana_regions.to_crs(epsg=32630)
ghana_regions["area_km2"] = ghana_regions.geometry.area / 1e6
ghana_regions = ghana_regions.to_crs(epsg=4326)

# -----------------------------
# 2. Charger ton dataset
# -----------------------------
df = pd.read_csv("ghana_villages_clean.csv", engine="python", on_bad_lines="skip")
df = df.dropna(subset=["Latitude", "Longitude", "Population"])
df = df[(df["Latitude"] != 0) & (df["Longitude"] != 0)]

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326"
)

# -----------------------------
# 3. Jointure villages → régions
# -----------------------------
joined = gpd.sjoin(gdf, ghana_regions, how="left", predicate="within")
pop_region = joined.groupby("name")["Population"].sum().reset_index()
ghana_regions = ghana_regions.merge(pop_region, on="name", how="left")

# -----------------------------
# 4. Calcul de la densité
# -----------------------------
ghana_regions["density"] = ghana_regions["Population"] / ghana_regions["area_km2"]

# -----------------------------
# 5. Ville la plus peuplée par région
# -----------------------------
idx = joined.groupby("name")["Population"].idxmax()
top_city_by_region = joined.loc[idx][["Village", "Latitude", "Longitude", "name"]]

# -----------------------------
# 6. Palette 8 couleurs
# -----------------------------
colors = [
    "#ffffcc", "#ffeda0", "#fed976", "#feb24c",
    "#fd8d3c", "#fc4e2a", "#e31a1c", "#b10026"
]
cmap_custom = LinearSegmentedColormap.from_list("ghana_density", colors)

# -----------------------------
# 7. Classes manuelles (propre, stable)
# -----------------------------
bins = [0, 70, 91, 168, 476, 1125, np.inf]
labels = [
    "0 – 70",
    "70 – 91",
    "91 – 168",
    "168 – 476",
    "476 – 1125",
    "> 1125"
]

ghana_regions["density_class"] = pd.cut(
    ghana_regions["density"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# -----------------------------
# 8. Plot
# -----------------------------
fig, ax = plt.subplots(1, 1, figsize=(14, 14))

ghana_regions.plot(
    column="density_class",
    cmap=cmap_custom,
    linewidth=0.8,
    edgecolor="white",
    legend=True,
    ax=ax
)

# -----------------------------
# 9. Légende propre (aucune décimale)
# -----------------------------
leg = ax.get_legend()
leg.set_title("Densité (hab/km²)")

# On force les labels (au cas où GeoPandas en ajoute)
for t, lab in zip(leg.texts, labels):
    t.set_text(lab)

# -----------------------------
# 10. Noms des régions (au-dessus des points)
# -----------------------------
for _, row in ghana_regions.iterrows():
    centroid = row.geometry.centroid
    plt.text(
        centroid.x,
        centroid.y + 0.35,   # décalage vertical
        row["name"],
        fontsize=10,
        ha="center",
        weight="bold",
        color="black",
        zorder=20
    )

# -----------------------------
# 11. Ville la plus peuplée par région
# -----------------------------
plt.scatter(
    top_city_by_region["Longitude"],
    top_city_by_region["Latitude"],
    s=90,
    color="black",
    zorder=5
)

for _, row in top_city_by_region.iterrows():
    plt.text(
        row["Longitude"] + 0.05,
        row["Latitude"] + 0.05,
        row["Village"],
        fontsize=9,
        color="black",
        zorder=30
    )

plt.title("Densité de population du Ghana (WorldPop 2020)", fontsize=18)
plt.axis("off")
plt.show()




In [ ]:
import folium
import geopandas as gpd
import pandas as pd
import numpy as np

# -----------------------------
# 1. Charger les régions du Ghana
# -----------------------------
admin1 = gpd.read_file(
    "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_1_states_provinces.zip"
)
ghana_regions = admin1[admin1["admin"] == "Ghana"].copy()

ghana_regions = ghana_regions.to_crs(epsg=32630)
ghana_regions["area_km2"] = ghana_regions.geometry.area / 1e6
ghana_regions = ghana_regions.to_crs(epsg=4326)

# -----------------------------
# 2. Charger ton dataset
# -----------------------------
df = pd.read_csv("ghana_villages_clean.csv", engine="python", on_bad_lines="skip")
df = df.dropna(subset=["Latitude", "Longitude", "Population"])
df = df[(df["Latitude"] != 0) & (df["Longitude"] != 0)]

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326"
)

# -----------------------------
# 3. Jointure villages → régions
# -----------------------------
joined = gpd.sjoin(gdf, ghana_regions, how="left", predicate="within")
pop_region = joined.groupby("name")["Population"].sum().reset_index()
ghana_regions = ghana_regions.merge(pop_region, on="name", how="left")

# -----------------------------
# 4. Densité
# -----------------------------
ghana_regions["density"] = ghana_regions["Population"] / ghana_regions["area_km2"]

# -----------------------------
# 5. Ville la plus peuplée
# -----------------------------
idx = joined.groupby("name")["Population"].idxmax()
top_city_by_region = joined.loc[idx][["Village", "Latitude", "Longitude", "name"]]

# -----------------------------
# 6. Classes manuelles
# -----------------------------
bins = [0, 70, 91, 168, 476, 1125, np.inf]
labels = [
    "0 – 70",
    "70 – 91",
    "91 – 168",
    "168 – 476",
    "476 – 1125",
    "> 1125"
]

ghana_regions["density_class"] = pd.cut(
    ghana_regions["density"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# -----------------------------
# 7. Palette 8 couleurs
# -----------------------------
colors = [
    "#ffffcc", "#ffeda0", "#fed976", "#feb24c",
    "#fd8d3c", "#fc4e2a", "#e31a1c", "#b10026"
]

color_map = dict(zip(labels, colors[:len(labels)]))

# -----------------------------
# 8. Carte Folium
# -----------------------------
m = folium.Map(location=[7.95, -1.02], zoom_start=7)

# -----------------------------
# 9. Polygones colorés
# -----------------------------
folium.GeoJson(
    ghana_regions,
    style_function=lambda feature: {
        "fillColor": color_map[feature["properties"]["density_class"]],
        "color": "white",
        "weight": 1,
        "fillOpacity": 0.8,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "density_class"],
        aliases=["Région", "Densité"],
        localize=True
    )
).add_to(m)

# -----------------------------
# 10. Ajouter les villes les plus peuplées (point + nom décalé)
# -----------------------------
for _, row in top_city_by_region.iterrows():
    # Point noir
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=5,
        color="black",
        fill=True,
        fill_color="black",
        popup=row["Village"]
    ).add_to(m)

    # Nom de la ville (décalé encore plus vers le haut)
    folium.Marker(
        location=[row["Latitude"] + 0.10, row["Longitude"]],   # décalage augmenté
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size:11px;
                font-weight:bold;
                color:black;
                text-shadow: 1px 1px 2px white;
                ">
                {row['Village']}
            </div>
            """
        )
    ).add_to(m)

# -----------------------------
# 11. Légende personnalisée
# -----------------------------
legend_html = """
<div style="position: fixed; 
     bottom: 30px; left: 30px; width: 180px; 
     background-color: white; z-index:9999; 
     padding: 10px; border:2px solid grey;">
<b>Densité (hab/km²)</b><br>
<div style="line-height: 18px;">
"""

for lab, col in color_map.items():
    legend_html += f"""
    <i style="background:{col};width:18px;height:18px;float:left;margin-right:8px;opacity:0.8"></i>{lab}<br>
    """

legend_html += "</div></div>"

m.get_root().html.add_child(folium.Element(legend_html))

# -----------------------------
# 12. Afficher la carte
# -----------------------------
m

m.save("ghana_density_map.html")
